## Ingrediant Parser 

In [1]:
from pydantic import BaseModel
from typing import List
import os 

from llama_index.core.program import FunctionCallingProgram
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.program import FunctionCallingProgram
from llama_index.llms.openai import OpenAI
from llama_index.readers.web import FireCrawlWebReader
from llama_index.core.agent.workflow import FunctionAgent, ReActAgent
from llama_index.core.program import LLMTextCompletionProgram


In [ ]:
os.environ["OPENAI_API_KEY"] =""
os.environ["GOOGLE_API_KEY"]=""

In [3]:
llm = OpenAI(model="gpt-4o-mini")
llm_gemini = GoogleGenAI(model="gemini-2.0-flash")


In [4]:
from llama_index.readers.web import FireCrawlWebReader

firecrawl_reader = FireCrawlWebReader(
    api_key="fc-e2c30d89a6164130ab16709c17429a02",
    mode="scrape",
    params={
        'formats': ['markdown'],
        "onlyMainContent":True,
        
    }
)

url_product = "https://www.amazon.com/s?k=ingredients"
url = "https://www.allrecipes.com/recipe/89268/triple-dipped-fried-chicken/"
docs = firecrawl_reader.load_data(url=url)  # ✅ single URL
docs_product= firecrawl_reader.load_data(url=url_product)  # ✅ single URL



In [5]:
docs[0].text

"\u200b\n\n[iframe](about:blank)\n\n# Triple-Dipped Fried Chicken\n\n4.4\n\n(983)\n\n773 Reviews\n\n195 Photos\n\nThis fried chicken batter yields the crispiest, spiciest, homemade fried chicken I have ever tasted! It has been a picnic favorite in my family for years and tastes equally good served hot or cold.\n\nSubmitted byQUIRKYIQ\n\nUpdated on April 15, 2024\n\nSave\n\nRate\n\nPrint\n\nShare\n\nAdd Photo\n[195\\\\\n![A blue tray with fried chicken and a small cup of dipping sauce](https://www.allrecipes.com/thmb/rwZ_WzbmuS8R7ZRlYwmBLDr8wdg=/160x90/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/AR-89268-triple-dipped-fried-chicken-beauty-4x3-3961ac838ddd41958e7cb9f49376cd68.jpg)](https://www.allrecipes.com/recipe/89268/triple-dipped-fried-chicken/#) [195\\\\\n![](https://imagesvc.meredithcorp.io/v3/mm/image?url=https%3A%2F%2Fpublic-assets.meredithcorp.io%2F6b0c98f2d1650d384d94dafd9dfb231b%2F1732723618894IMG_20241127_100424783.jpg&w=160&q=60&c=sc&poi=auto&orient=true

In [6]:
from pydantic import BaseModel, HttpUrl
from typing import List, Optional


class Ingredient(BaseModel):
    """Structured information for a single ingredient."""
    name: str
    category: Optional[str] = None
    quantity: Optional[str] = None  # Accepts flexible formats like "1 ½ cups", "2 tbsp"


class Step(BaseModel):
    """A single step in the cooking process."""
    order: int
    instruction: str


class NutritionInfo(BaseModel):
    """Nutritional details per serving."""
    calories: Optional[str] = None
    fat: Optional[str] = None
    carbs: Optional[str] = None
    protein: Optional[str] = None


class ExtractRecipe(BaseModel):
    """Comprehensive schema for a recipe."""
    title: str
    description: Optional[str]
    ingredients: List[Ingredient]
    steps: List[Step]
    servings: Optional[int]
    prep_time: Optional[str]
    cook_time: Optional[str]
    total_time: Optional[str]
    image_url: Optional[str]
    video_url: Optional[str]
    source_url: Optional[str]
    author: Optional[str]
    cuisine: Optional[str]
    tags: Optional[List[str]]
    nutrition: Optional[NutritionInfo]
    notes: Optional[List[str]]


recipe_prompt = """\
You are a structured data extraction agent. You MUST return data using the provided function schema. Do NOT return freeform text.
You are a helpful assistant that extracts detailed recipe data from the given text. 
The content comes from a cooking blog or recipe website and may contain editorial content, tips, and reviews.

Your job is to extract structured information and return it as a JSON object with the following fields:

Required:
- title: The name of the recipe.
- description: A short summary describing the dish.
- ingredients: A list of ingredients. Each ingredient must include:
  - name: The name of the ingredient (e.g., "all-purpose flour").
  - category: The type/category of the ingredient (e.g., "dairy", "spice", "vegetable"). Leave empty if uncertain.
  - quantity: The amount (e.g., "1 ½ cups", "2 tablespoons", "a pinch"). Leave empty if not specified.

- steps: A list of cooking instructions in order. Each step should include:
  - order: Step number starting from 1.
  - instruction: A single, clear action from the recipe.

Optional:
- servings: Number of servings the recipe makes.
- prep_time: Time to prepare ingredients (e.g., "15 minutes").
- cook_time: Time to cook (e.g., "20 minutes").
- total_time: Combined prep and cook time if available.
- image_url: URL of the main recipe image, if available.
- video_url: URL of a recipe video, if available.
- source_url: Original recipe URL if mentioned.
- author: Name of the person who submitted or wrote the recipe.
- cuisine: The cuisine type (e.g., "Italian", "Middle Eastern").
- tags: Keywords or categories mentioned (e.g., "easy", "low-carb").
- nutrition: Nutrition information per serving if available. Include:
  - calories
  - fat
  - carbs
  - protein
- notes: A list of extra tips, editor remarks, or serving suggestions mentioned in the recipe content.

Important Instructions:
- Avoid extracting editorial content, reviews, or advertisements.
- Focus only on the actual recipe content and ignore unrelated sections.
- If the recipe steps contain sub-steps (e.g., make batter, then fry), preserve this logically in the ordered steps as separate entries.
- If any required fields are missing or cannot be determined, leave them empty.
Here is the text:
{input_text}

Return the result as a well-structured JSON object that matches the field names exactly.
"""


recipe_extractor = LLMTextCompletionProgram.from_defaults(
    output_cls=ExtractRecipe,
    prompt_template_str=recipe_prompt,
    verbose=True,
    llm=llm_gemini,
  )

In [7]:
class ExtractProductItem(BaseModel):
    name: str
    category: str
    quantity: int
    price: float

class ExtractProductList(BaseModel):
    products: List[ExtractProductItem]


product_prompt = """
You are a helpful assistant that extracts product information from a given text.

Here is the text:
{input_text}

Please extract **all products** mentioned in the text. For each product, extract the following fields:
- name
- category
- quantity
- price

Respond with a JSON array in the following format:
[
  {
    "name": "...",
    "category": "...",
    "quantity": ...,
    "price": ...
  },
  ...
]

Make sure all objects are valid and follow the schema. Only include products that contain at least a name and category.
"""

llm = OpenAI(model="gpt-4o-mini")

product_extractor = LLMTextCompletionProgram.from_defaults(
    output_cls=ExtractProductList,
    prompt_template_str=product_prompt,
    verbose=True,
    llm=llm_gemini,
)

In [8]:
from llama_index.core.prompts import PromptTemplate
from enum import Enum

class ContentEnum(str, Enum):
    recipe = "recipe"
    product = "product"
    other = "other"

class WebClassifier(BaseModel):
    type: ContentEnum

CLASSIFICATION_PROMPT = PromptTemplate("""
You are a smart content classifier for web pages.

Your job is to classify the **primary purpose** of the page as one of:

- "recipe": if the main purpose of the content is to teach the reader how to prepare or cook a meal. This includes step-by-step instructions, ingredient lists, and cooking methods. It may mention product brands or links to purchase items, but the focus should be on preparing a dish.

- "product": if the main purpose of the content is to present, describe, compare, review, or sell one or more products. This includes product listings, ecommerce pages, shopping guides, or top-10 comparisons. The content may mention how a product is used in cooking, but it does not provide a full recipe.

- "other": if the content does not fall into either category above. This may include blog posts, opinion pieces, lifestyle articles, or content unrelated to food or products.

Return ONLY the type.

Content:
{text}
""")


In [9]:
result = llm_gemini.structured_predict(WebClassifier, CLASSIFICATION_PROMPT, text=docs[0].text)

print(result)

type=<ContentEnum.recipe: 'recipe'>


In [10]:
result.type.value

'recipe'

In [11]:
def classify_web_content(text: str):
    result = llm_gemini.structured_predict(WebClassifier, CLASSIFICATION_PROMPT, text=text)
    return result.type


def extract_recipe(text: str):
    return recipe_extractor(input_text=text)

def extract_products(text: str):
    return product_extractor(input_text=text)

def read_url_content(url: str):
    docs = firecrawl_reader.load_data(url=url)
    return docs[0].text if docs else ""





In [14]:
def extract_ingredient(url: str):
    """
    Extracts ingredients or product information from the given URL.
    Automatically classifies the page as either 'recipe' or 'product' and invokes the appropriate extractor.
    """
    page_content = read_url_content(url)

    if not page_content:
        print("No content found at the provided URL.")

    classification = classify_web_content(page_content)

    print(f"Content classified as: {classification}")

    if classification == ContentEnum.recipe:
        result = extract_recipe(page_content)
        return {
            "type": "recipe",
            "data": result.model_dump()  # or result.dict()
        }

    elif classification == ContentEnum.product:
        result = extract_products(page_content)
        return {
            "type": "product",
            "data": result.model_dump()
        }

    else:
        print.warning(f"Unclassified content for URL: {url}")
        return {
            "type": "other",
            "data": {}
        }

In [15]:
url = "https://www.allrecipes.com/recipe/89268/triple-dipped-fried-chicken/"
output = extract_ingredient(url)

Content classified as: ContentEnum.recipe


In [16]:
output

{'type': 'recipe',
 'data': {'title': 'Triple-Dipped Fried Chicken',
  'description': 'This fried chicken batter yields the crispiest, spiciest, homemade fried chicken I have ever tasted! It has been a picnic favorite in my family for years and tastes equally good served hot or cold.',
  'ingredients': [{'name': 'vegetable oil',
    'category': 'condiment',
    'quantity': '1 quart'},
   {'name': 'all-purpose flour',
    'category': 'grain',
    'quantity': '4 ⅓ cups, divided'},
   {'name': 'garlic salt', 'category': 'spice', 'quantity': '1 ½ tablespoons'},
   {'name': 'ground black pepper',
    'category': 'spice',
    'quantity': '1 tablespoon'},
   {'name': 'paprika', 'category': 'spice', 'quantity': '1 tablespoon'},
   {'name': 'poultry seasoning',
    'category': 'spice',
    'quantity': '½ teaspoon'},
   {'name': 'beer',
    'category': 'beverage',
    'quantity': '1 ½ cups, or as needed'},
   {'name': 'egg yolks', 'category': 'dairy', 'quantity': '2, beaten'},
   {'name': 'salt'